# 02 — Nettoyage des données · Databricks
**Exécuter après** : `01_exploration_db.ipynb`
---

In [ ]:
%pip install vaderSentiment -q

In [ ]:
# ============================================================
# Setup Databricks — import spark_utils par chemin absolu
# Évite tout conflit avec le package pip "spark_utils" system
# ============================================================
import sys, os, importlib.util

# 1. Résolution du repo root via Databricks Repos context
_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_parts = _ctx.split('/')
_idx   = next((i for i, p in enumerate(_parts) if p == 'fakenews-analyzer'), None)

if _idx is None:
    raise RuntimeError(f"'fakenews-analyzer' introuvable dans le path Databricks : {_ctx}")

_REPO       = '/Workspace/' + '/'.join(_parts[1:_idx+1])
_UTILS_FILE = _REPO + '/02_preprocessing/spark_utils.py'

if not os.path.exists(_UTILS_FILE):
    raise FileNotFoundError(
        f"spark_utils.py introuvable : {_UTILS_FILE}
"
        "Vérifier que le repo est bien lié dans Databricks Repos et que le Pull est fait."
    )

# 2. Import direct par fichier (bypass total du sys.path et des packages pip)
_spec = importlib.util.spec_from_file_location('_spark_utils', _UTILS_FILE)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

load_raw_sources        = _mod.load_raw_sources
show_label_distribution = _mod.show_label_distribution
save_parquet            = _mod.save_parquet
load_parquet            = _mod.load_parquet
stratified_split        = _mod.stratified_split
check_class_balance     = _mod.check_class_balance

# 3. Chemins Volume Unity Catalog
# → Databricks > Data > Volumes > clic droit "fakenews" > Copy path
VOLUME        = '/Volumes/main/default/fakenews'  # <- MODIFIER si catalog/schema différent

RAW_DIR       = VOLUME + '/raw'
PROCESSED_DIR = VOLUME + '/processed'
MODELS_DIR    = VOLUME + '/models'
COLAB_DIR     = VOLUME + '/colab_exports'
REPORTS_DIR   = VOLUME + '/reports'

for d in [PROCESSED_DIR, MODELS_DIR, COLAB_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Repo       : {_REPO}')
print(f'spark_utils: {_UTILS_FILE}  ✓')
print(f'Volume     : {VOLUME}')
print('Setup OK')

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import pandas as pd

sources = load_raw_sources(spark, RAW_DIR)
print(f'Sources disponibles : {list(sources.keys())}')

## Section 1 — Fonction de nettoyage

In [ ]:
import re

ENGLISH_STOPWORDS = [
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','will','would','could','should','may','might','shall','can','need',
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'he','him','his','himself','she','her','hers','herself','it','its','itself',
    'they','them','their','theirs','themselves','what','which','who','whom',
    'this','that','these','those','am','if','else','each','both','few','more',
    'most','other','some','such','no','nor','not','only','own','same','so',
    'than','too','very','just','because','as','until','while','about','against',
    'between','into','through','during','before','after','above','below','from',
    'up','down','out','off','over','under','again','further','then','once'
]

def clean_text_python(text) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    tokens = [t for t in text.split() if t not in ENGLISH_STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

clean_udf = F.udf(clean_text_python, StringType())
print('Fonction de nettoyage definie.')

## Section 2 — Application du nettoyage

In [ ]:
cleaned_sources = {}

for name, df in sources.items():
    print(f'\n── Nettoyage : {name} ──')
    text_col  = next((c for c in df.columns if c.lower() in ('text','statement','tweet','content')), None)
    label_col = next((c for c in df.columns if c.lower() in ('label','fraudulent')), None)
    if not text_col:
        print(f'  Pas de colonne texte — ignore')
        continue
    before = df.count()
    print(f'  Lignes avant : {before:,}')
    df_clean = df.withColumn('clean_text', clean_udf(F.col(text_col)))
    keep_cols = ['clean_text']
    if label_col:
        keep_cols.append(label_col)
        if label_col != 'label':
            df_clean = df_clean.withColumnRenamed(label_col, 'label')
            keep_cols = ['clean_text', 'label']
    df_clean = df_clean.select(keep_cols)
    cleaned_sources[name] = df_clean
    print(f'  OK | colonnes : {df_clean.columns}')

## Section 3 — Suppression des lignes vides

In [ ]:
for name in list(cleaned_sources.keys()):
    df = cleaned_sources[name]
    before = df.count()
    df = df.filter(F.col('clean_text').isNotNull())
    df = df.filter(F.trim(F.col('clean_text')) != '')
    df = df.filter(F.size(F.split(F.col('clean_text'), ' ')) >= 5)
    after = df.count()
    print(f'[{name}] Vides supprimes : {before - after:,} -> {after:,} lignes')
    cleaned_sources[name] = df

## Section 4 — Suppression des doublons

In [ ]:
for name in list(cleaned_sources.keys()):
    df = cleaned_sources[name]
    before = df.count()
    df = df.dropDuplicates(['clean_text'])
    after = df.count()
    print(f'[{name}] Doublons supprimes : {before - after:,} -> {after:,} lignes uniques')
    cleaned_sources[name] = df

## Section 5 — Stats après nettoyage

In [ ]:
stats = []
for name, df in cleaned_sources.items():
    total = df.count()
    if 'label' in df.columns:
        label_dist = df.groupBy('label').count().toPandas()
        fake_count = int(label_dist[label_dist['label'] == 1]['count'].sum()) if 1 in label_dist['label'].values else 0
        real_count = int(label_dist[label_dist['label'] == 0]['count'].sum()) if 0 in label_dist['label'].values else 0
    else:
        fake_count = real_count = 'N/A'
    avg_len = df.select(F.avg(F.size(F.split(F.col('clean_text'), ' '))).alias('avg')).collect()[0]['avg']
    stats.append({'source': name, 'total': total, 'FAKE': fake_count, 'REAL': real_count,
                  'avg_words': round(avg_len, 1) if avg_len else 0})
display(pd.DataFrame(stats))

## Section 6 — Sauvegarde en Parquet

In [ ]:
for name, df in cleaned_sources.items():
    output_path = os.path.join(PROCESSED_DIR, f'cleaned_{name}.parquet')
    df_with_source = df.withColumn('source', F.lit(name))
    save_parquet(df_with_source, output_path)

print(f'\nTous les fichiers nettoyes sauvegardes dans {PROCESSED_DIR}')
print('Prochaine etape : 03_unification_db.ipynb')